## Pyspark vs SQL
🧠Regla mental que debe repetir en clase
- SQL = declarativo
- Pyspark = transformaciones encadenadas sobre dataframes

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_customers
LIMIT 5

In [0]:
df = spark.read.table('samples.bakehouse.sales_customers')

df_selected = df.select('*')

df_selected.display()

In [0]:
%sql
SELECT DISTINCT country
FROM samples.bakehouse.sales_customers


In [0]:
df_distinct = df_selected.select('country').distinct()
display(df_distinct)

In [0]:
%sql
SELECT *
FROM samples.bakehouse.sales_customers
WHERE 1=1
AND country = 'Japan'
AND gender = 'female'
LIMIT(10)

In [0]:
from pyspark.sql.functions import col
df_filtered = df_selected.filter((col('country')=='USA') &
                                 (col('gender')=='female'))
display(df_filtered)

In [0]:
df_sorted = df.orderBy(col('first_name'))
display(df_sorted.limit(4))

In [0]:
%sql
SELECT
    'HOLA' AS SALUDO

In [0]:
%sql
SELECT
    concat(first_name," ",last_name) AS full_name,
    concat_ws(', ',city, state, country, continent) AS adress,
    35 AS age,
    'random coment' AS comment,
    true AS is_true
FROM samples.bakehouse.sales_customers
LIMIT 10


In [0]:
from pyspark.sql.functions import col, concat, lit
df_plus_columns = df.select(
    concat(col('first_name'), lit(' '), col('last_name')).alias('full_name'),
    lit(True).alias('is_true'),
    lit('random comment').alias('comment')
).limit(10)
display(df_plus_columns)

In [0]:
%sql
SELECT country, count(1) AS cantidad
FROM samples.bakehouse.sales_franchises
GROUP BY country
ORDER BY cantidad DESC

In [0]:
from pyspark.sql.functions import count

df_grouped = (
    df
    .groupby("country")
    .agg(count("*").alias("count"))
    .orderBy(col("count").desc())
)
display(df_grouped)

In [0]:
%sql
SELECT
    concat(c.first_name, " ", c.last_name) AS full_name,
    c.country,
    c.gender,
    s.product,
    s.quantity,
    s.paymentMethod,
    s.dateTime
FROM samples.bakehouse.sales_transactions s
JOIN samples.bakehouse.sales_customers c
ON s.customerID = c.customerID;

In [0]:
df_sales = spark.read.table("samples.bakehouse.sales_transactions")

df_joined = (df
             .join(df_sales, on="customerID", how="inner")
             .select("*")
             )
df_joined.display()

In [0]:
%sql
SELECT
    CAST(transactionID AS STRING) AS transactionID,
    CAST(customerID AS STRING) AS customerID,
    CAST(franchiseID AS STRING) AS franchiseID,
    CAST(cardNumber AS STRING) AS cardNumber
FROM samples.bakehouse.sales_transactions

In [0]:
from pyspark.sql.functions import col
df_casted = df_sales.select(
    col("transactionID").cast("long"),
    col("customerID").cast("long"),
    col("franchiseID").cast("long")
)

display(df_casted)

In [0]:
df_test = df_casted.withColumn("nuevaColumna",col("transactionID").cast("bigint"))

df_new = df_test.drop("nuevaColumna")
display(df_new)

        